# Lección 01 - Introducción a los Agentes de IA

¡Bienvenido a la primera lección del curso **Agentes de IA para Principiantes**!

Un **agente de IA** es un programa que utiliza un modelo de lenguaje grande (LLM) como su motor de razonamiento y puede tomar *acciones* en el mundo real — llamando a APIs, consultando bases de datos o ejecutando código — para cumplir un objetivo en nombre de un usuario.

En este cuaderno construirás tu primer agente: un **Agente de Viajes** que recomienda destinos de vacaciones. En el proceso aprenderás a:

1. Conectarte al Servicio Azure AI Foundry Agent usando el **Microsoft Agent Framework**.
2. Darle al agente una **herramienta** — una función Python simple que puede llamar.
3. Ejecutar el agente e inspeccionar su respuesta.
4. Transmitir la respuesta del agente token por token.


## Configuración

Antes de ejecutar este cuaderno, asegúrate de tener:

1. **Un proyecto de Azure AI Foundry** con un modelo de chat desplegado (por ejemplo, `gpt-4o-mini`).
2. **Sesión iniciada con Azure CLI** — ejecuta `az login` en tu terminal.
3. **Configuradas las variables de entorno necesarias:**
   - `AZURE_AI_PROJECT_ENDPOINT` — el endpoint de tu proyecto de Azure AI Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — el nombre de tu modelo desplegado.

La celda siguiente instala los paquetes de Python que necesitas.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Creando Tu Primer Agente

Un agente necesita dos cosas:

- **Instrucciones** que le digan *quién es* y *cómo debe comportarse* (un prompt del sistema).
- **Herramientas** — funciones de Python decoradas con `@tool` que el agente puede llamar para obtener información o realizar acciones.

Abajo definimos una herramienta simple que devuelve una lista de destinos populares para vacaciones. El agente usará esta herramienta cuando un usuario pida recomendaciones de viajes.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Respuestas en streaming

Para una experiencia más interactiva, puedes **transmitir** la respuesta del agente. En lugar de esperar la respuesta completa, el agente entrega fragmentos de texto a medida que se generan. Esto es especialmente útil en interfaces de chat donde quieres mostrar la salida en tiempo real.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Summary

En esta lección aprendiste a:

- **Crear un proveedor** que se conecta al servicio Azure AI Foundry Agent mediante `AzureAIProjectAgentProvider`.
- **Definir una herramienta** usando el decorador `@tool` para que el agente pueda llamar a tus funciones de Python.
- **Ejecutar el agente** con un mensaje del usuario e imprimir su respuesta.
- **Transmitir respuestas** para una salida en tiempo real.

En la próxima lección exploraremos los frameworks agentic con más profundidad y aprenderemos cómo dotar a los agentes de herramientas más potentes y capacidades de razonamiento en múltiples pasos.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Descargo de responsabilidad**:
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automatizadas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional humana. No somos responsables de cualquier malentendido o interpretación errónea que surja del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
